#Fine-Tuning OpenAI Models

Copyright 2024,2025,2026 Denis Rothman

[OpenAI fine-tuning documentation](https://beta.openai.com/docs/guides/fine-tuning/)

Check the cost of fine-tuning your dataset on OpenAI before running the notebook.

Run this notebook cell by cell to:

1.Download and prepare the data   
2.Fine-tune a model   
3.Run a fine-tuned model

# Installing the environment


In [1]:
try:
  import openai
except:
  !pip install openai==1.45.0
  import openai

In [2]:
import os
from openai import OpenAI
from google.colab import userdata

try:
    # Explicitly check for the mandatory OpenAI key
    openai_key = userdata.get("API_KEY")
    if not openai_key:
        raise ValueError("OpenAI API_KEY not found in Secrets.")

    os.environ["OPENAI_API_KEY"] = openai_key
    client = OpenAI()
    print("✅ OpenAI initialized (Mandatory).")

except Exception as e:
    print(f"❌ Critical Error: OpenAI initialization failed. {e}")
    # We raise the error here because OpenAI is required for the Planner to work at all
    raise e

✅ OpenAI initialized (Mandatory).


In [3]:
!pip install jsonlines==4.0.0

In [4]:
!pip install -U datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 13.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


# 1.Preparing the dataset for fine-tuning

## 1.1.Downloading and displaying the dataset

In [ ]:
import pandas as pd

# 1. Download the data directly from Hugging Face (bypassing the failing library)
url = "https://huggingface.co/datasets/allenai/sciq/resolve/main/data/train-00000-of-00001.parquet"
df = pd.read_parquet(url)

# 2. Filter the dataset to include only questions with support and correct answer
# (Using Pandas syntax instead of datasets syntax)
filtered_dataset = df[(df["support"] != "") & (df["correct_answer"] != "")]

# 3. Print the number of questions (Validation)
print("Number of questions with support: ", len(filtered_dataset))

# Note: The variable 'filtered_dataset' is now a Pandas DataFrame.
# The next cell in your notebook (df_view = pd.DataFrame(filtered_dataset))
# will still work perfectly fine with this change.

Number of questions with support:  10481


In [ ]:
# Convert the filtered dataset to a pandas DataFrame
df_view = pd.DataFrame(filtered_dataset)

# Columns to drop
columns_to_drop = ['distractor3', 'distractor1', 'distractor2']

# Dropping the columns from the DataFrame
df_view = df_view.drop(columns=columns_to_drop)

# Display the DataFrame
df_view.head()

,question,correct_answer,support
0,What type of organism is commonly used in prep...,mesophilic organisms,"Mesophiles grow best in moderate temperature, ..."
1,What phenomenon makes global winds blow northe...,coriolis effect,Without Coriolis Effect the global winds would...
2,Changes from a less-ordered state to a more-or...,exothermic,Summary Changes of state are examples of phase...
3,What is the least dangerous radioactive decay?,alpha decay,All radioactive decay is dangerous to living t...
4,Kilauea in hawaii is the world’s most continuo...,smoke and ash,Example 3.5 Calculating Projectile Motion: Hot...


## 1.2. Preparing the dataset for fine-tuning

In [ ]:
import json
import jsonlines
import pandas as pd

# 1. We use the 'df_view' dataframe you created in the previous successful cells.
#    This contains the exact same data as the 'dataset' variable from the failing code.

items = []

# 2. We iterate through the dataframe just like the original code
for idx, row in df_view.iterrows():
    # Exact same formatting logic as the original code
    detailed_answer = row['correct_answer'] + " Explanation: " + row['support']

    items.append({
        "messages": [
            {"role": "system", "content": "Given a science question, provide the correct answer with a detailed explanation."},
            {"role": "user", "content": row['question']},
            {"role": "assistant", "content": detailed_answer}
        ]
    })

# 3. Write to the exact same filename using the same library
output_file = '/content/QA_prompts_and_completions.json'
with jsonlines.open(output_file, 'w') as writer:
    writer.write_all(items)

print(f"✅ Successfully created {output_file} with {len(items)} examples.")

✅ Successfully created /content/QA_prompts_and_completions.json with 10481 examples.


### Visualizing the JSON file

In [ ]:
dfile="/content/QA_prompts_and_completions.json"

In [ ]:
import pandas as pd

# Load the data
df = pd.read_json(dfile, lines=True)
df

,messages
0,"[{'role': 'system', 'content': 'Given a scienc..."
1,"[{'role': 'system', 'content': 'Given a scienc..."
2,"[{'role': 'system', 'content': 'Given a scienc..."
3,"[{'role': 'system', 'content': 'Given a scienc..."
4,"[{'role': 'system', 'content': 'Given a scienc..."
...,...
10476,"[{'role': 'system', 'content': 'Given a scienc..."
10477,"[{'role': 'system', 'content': 'Given a scienc..."
10478,"[{'role': 'system', 'content': 'Given a scienc..."
10479,"[{'role': 'system', 'content': 'Given a scienc..."


# 2.Fine-tuning the model



https://platform.openai.com/docs/guides/supervised-fine-tuning


In [ ]:
from openai import OpenAI
import jsonlines
client = OpenAI()
# Uploading the training file

result_file = client.files.create(
  file=open("QA_prompts_and_completions.json", "rb"),
  purpose="fine-tune"
)

print(result_file)
param_training_file_name = result_file.id
print(param_training_file_name)

# Creating the fine-tuning job

ft_job = client.fine_tuning.jobs.create(
  training_file=param_training_file_name,
  model="gpt-4.1-mini-2025-04-14"
)

# Printing the fine-tuning job
print(ft_job)

FileObject(id='file-FZBf7WdUp8s3hmiCpgo2TC', bytes=8062970, created_at=1768040390, filename='QA_prompts_and_completions.json', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)
file-FZBf7WdUp8s3hmiCpgo2TC
FineTuningJob(id='ftjob-JqLq1qdQ5Vo2jWQFhGL1IUoq', created_at=1768040392, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto'), model='gpt-4.1-mini-2025-04-14', object='fine_tuning.job', organization_id='org-h2Kjmcir4wyGtqq1mJALLGIb', result_files=[], seed=337058542, status='validating_files', trained_tokens=None, training_file='file-FZBf7WdUp8s3hmiCpgo2TC', validation_file=None, estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size='auto', learning_rate_multipli

# 3.Monitoring the fine-tunes

In [5]:
import pandas as pd
from openai import OpenAI
client = OpenAI()
# Assume client is already set up and authenticated
response = client.fine_tuning.jobs.list(limit=3) # increase to include your history

# Initialize lists to store the extracted data
job_ids = []
created_ats = []
statuses = []
models = []
training_files = []
error_messages = []
fine_tuned_models = []  # List to store the fine-tuned model names

# Iterate over the jobs in the response
for job in response.data:
    job_ids.append(job.id)
    created_ats.append(job.created_at)
    statuses.append(job.status)
    models.append(job.model)
    training_files.append(job.training_file)
    error_message = job.error.message if job.error else None
    error_messages.append(error_message)

    # Append the fine-tuned model name
    fine_tuned_model = job.fine_tuned_model if hasattr(job, 'fine_tuned_model') else None
    fine_tuned_models.append(fine_tuned_model)

# Create a DataFrame
df = pd.DataFrame({
    'Job ID': job_ids,
    'Created At': created_ats,
    'Status': statuses,
    'Model': models,
    'Training File': training_files,
    'Error Message': error_messages,
    'Fine-Tuned Model': fine_tuned_models  # Include the fine-tuned model names
})

# Convert timestamps to readable format
df['Created At'] = pd.to_datetime(df['Created At'], unit='s')
df = df.sort_values(by='Created At', ascending=False)

# Display the DataFrame
df

,Job ID,Created At,Status,Model,Training File,Error Message,Fine-Tuned Model
0,ftjob-JqLq1qdQ5Vo2jWQFhGL1IUoq,2026-01-10 10:19:52,succeeded,gpt-4.1-mini-2025-04-14,file-FZBf7WdUp8s3hmiCpgo2TC,None,ft:gpt-4.1-mini-2025-04-14:personal::CwQpKy8Y
1,ftjob-Cyvpg5kEZ4wAo3zKiQH7wI8P,2026-01-10 10:06:19,failed,gpt-4.1-mini-2025-04-14,file-WKCu1mukrTgLUWLfjEb4fM,Creating this fine-tuning job would exceed you...,None
2,ftjob-f9JxcdrNVYiVnO4M3EZkLaLT,2025-08-13 09:23:49,succeeded,gpt-4.1-mini-2025-04-14,file-6VgWqp3wmd6Tos6aUihLLZ,None,ft:gpt-4.1-mini-2025-04-14:personal::C42Vkm29


### Make sure to obtain your fine-tune model here

If your OpenAI notifications are activated you should receive an email.

Otherwise run the "Monitoring the fine-tunes" cell above to check the status of your fine-tune job.

In [8]:
import pandas as pd

generation=True  # False until the current model is fine-tuned, True when the current model is fine-tuned
# Attempt to find the first non-empty Fine-Tuned Model
non_empty_models = df[df['Fine-Tuned Model'].notna() & (df['Fine-Tuned Model'] != '')]

if not non_empty_models.empty:
    first_non_empty_model = non_empty_models['Fine-Tuned Model'].iloc[0]
    print("The latest fine-tuned model is:", first_non_empty_model)
    generation=True
else:
    first_non_empty_model='None'
    print("No fine-tuned models found.")

The latest fine-tuned model is: ft:gpt-4.1-mini-2025-04-14:personal::CwQpKy8Y


*Note:* Only continue to Step 3, to use the fine-tuned model when your fine-tuned model is ready. If your OpenAI notifications is activiated, you will receive an email with the status of your fine-tunning job.

# 4.Using the fine-tuned OpenAI model

Note: The is a fine-tuning. As such, be patient!
Rune the `Monitoring the fine-tunes` cell and the f`irst_non_empty_model` cell from time to time.

If the fine-tunning succeeded and your model is ready, the name of your model will be `first_non_empty_model`

1.Go to the OpenAI Playground to test your model: https://platform.openai.com/playground

2.Check the metrics in the fine-tuning UI:
https://platform.openai.com/finetune/

3.Try the fined-tune model out in the cell below.

In [9]:
# Define the prompt
prompt = "What phenomenon makes global winds blow northeast to southwest or the reverse in the northern hemisphere and northwest to southeast or the reverse in the southern hemisphere?"

*Note:* Only run the following cell if your fine-tune job has succeeded and a fined-tuned model is found in the *Monitoring the fine-tunes"* section of *2.Fine-tuning the model.*

In [10]:
# Assume first_non_empty_model is defined above this snippet
if generation==True:
    response = client.responses.create(
      model=first_non_empty_model,
      input=[
        {"role": "system", "content": "Given a question, reply with a complete explanation for students."},
        {"role": "user", "content": prompt}
     ],
     temperature=0.0,
     max_output_tokens=200
)
else:
    print("Error: Model is None, cannot proceed with the API request.")

In [11]:
if generation==True:
  print(response)

Response(id='resp_0aa8f7782af5df50006962812c67448193999db901c638fff7', created_at=1768063276.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='ft:gpt-4.1-mini-2025-04-14:personal::CwQpKy8Y', object='response', output=[ResponseOutputMessage(id='msg_0aa8f7782af5df5000696281310d10819387c43b29b4675898', content=[ResponseOutputText(annotations=[], text='coriolis effect Explanation: Global winds blow northeast to southwest or the reverse in the Northern Hemisphere. In the Southern Hemisphere, they blow northwest to southeast or the reverse. This is due to the Coriolis effect.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message')], parallel_tool_calls=True, temperature=0.0, tool_choice='auto', tools=[], top_p=1.0, background=False, conversation=None, max_output_tokens=200, max_tool_calls=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_retention=None, reasoning=Reasoning(effort=None, generate_su

In [12]:
if (generation==True):
  # Access the response from the first choice
  response_text = response.output_text
  # Print the response
  print(response_text)

coriolis effect Explanation: Global winds blow northeast to southwest or the reverse in the Northern Hemisphere. In the Southern Hemisphere, they blow northwest to southeast or the reverse. This is due to the Coriolis effect.


In [13]:
import textwrap

if generation==True:
  wrapped_text = textwrap.fill(response_text.strip(), 60)
  print(wrapped_text)

coriolis effect Explanation: Global winds blow northeast to
southwest or the reverse in the Northern Hemisphere. In the
Southern Hemisphere, they blow northwest to southeast or the
reverse. This is due to the Coriolis effect.


[Consult OpenAI fine-tune documentation for more](https://platform.openai.com/docs/guides/fine-tuning)